# Candidate-Selection Approaches — Local Test Prototypes

Working, self-contained prototypes of the five re-prioritized candidate-selection
approaches, so the *logic* can be tested without standing up the MCP server.

- Data: local SQLite catalog (`data/db/inventaa_knowledge_base.db`).
- Embeddings: Azure OpenAI if a real embeddings endpoint is configured (embedded
  one string at a time), else a TF-IDF numpy baseline so the notebook always runs.
- These prove the algorithms. Productionizing moves them into the MCP server
  (e.g. `scope_skus` param, reranker step, feedback store, RAPTOR index).

Run `set-env.ps1` first so the catalog path / env vars are available.


In [1]:
# 0. Setup (SQLite only)
import os, re, json, sqlite3
from collections import Counter
from pathlib import Path
import numpy as np

# The notebook may be run from the project root or from `notebooks/`, which would
# otherwise resolve the catalog to an empty `notebooks/data/db/...` file. Point
# SQLITE_PATH at the first candidate DB that actually contains products so the
# prototype always loads real data regardless of the working directory.
def _find_catalog_db():
    here = Path(os.getcwd())
    candidates = [
        here / "data" / "db" / "inventaa_knowledge_base.db",
        here.parent / "data" / "db" / "inventaa_knowledge_base.db",
        here / "data" / "db" / "knowledge_base.db",
        here.parent / "data" / "db" / "knowledge_base.db",
    ]
    for c in candidates:
        if c.exists():
            try:
                con = sqlite3.connect(str(c))
                n = con.execute("SELECT COUNT(*) FROM products").fetchone()[0]
                con.close()
                if n > 0:
                    return str(c)
            except Exception:
                pass
    return None

catalog_db = os.getenv("SQLITE_PATH") or _find_catalog_db()
if catalog_db:
    os.environ["SQLITE_PATH"] = catalog_db
print("catalog db:", catalog_db)

from src.db.database import get_session
from src.db.models import Product



catalog db: c:\Users\jimso\Downloads\inventaa-graphrag\data\db\inventaa_knowledge_base.db


In [2]:
# 1. Embedder (Azure if a real embeddings endpoint is set, else TF-IDF baseline)
def make_embedder():
    # --- try Azure OpenAI embeddings (embed ONE string at a time; some Azure
    #     deployments reject array `input`, which causes a 400 "$.input invalid") ---
    try:
        from openai import AzureOpenAI
        ep = os.getenv("AZURE_OPENAI_ENDPOINT") or os.getenv("AZURE_AI_ENDPOINT")
        key = os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("AZURE_AI_API_KEY")
        dep = os.getenv("TEXT_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
        if ep and key:
            client = AzureOpenAI(api_key=key, azure_endpoint=ep,
                                  api_version=os.getenv("AZURE_AI_API_VERSION", "2024-12-01-preview"))
            def az(texts):
                if isinstance(texts, str): texts = [texts]
                vecs = []
                for t in texts:                      # one call per string -> avoids array rejection
                    o = client.embeddings.create(model=dep, input=t)
                    vecs.append(o.data[0].embedding)
                return np.array(vecs, dtype=np.float32)
            try:
                az("smoke-test")                     # verify the deployment serves embeddings
                print("embedder: Azure OpenAI (per-string)")
                return az
            except Exception as e:
                print("Azure embedder call failed -> TF-IDF fallback:", type(e).__name__, str(e)[:90])
    except Exception as e:
        print("Azure embedder unavailable -> TF-IDF fallback:", type(e).__name__, str(e)[:90])

    # --- TF-IDF baseline (always works, lexical not semantic) ---
    class Tfidf:
        def __init__(self, corpus):
            toks = [re.sub(r"[^a-z0-9 ]", " ", d.lower()).split() for d in corpus]
            df = Counter()
            for d in toks:
                for w in set(d): df[w] += 1
            vocab = sorted({w for d in toks for w in set(d) if len(w) > 1})
            self.w2i = {w: i for i, w in enumerate(vocab)}
            self.N = len(vocab)
            self.idf = np.array([np.log(len(toks) / max(1, df[w])) for w in vocab])
        def vec(self, text):
            t = [w for w in re.sub(r"[^a-z0-9 ]", " ", text.lower()).split() if w in self.w2i]
            v = np.zeros(self.N)
            for w, c in Counter(t).items():
                i = self.w2i[w]; v[i] = (1 + np.log(c)) * self.idf[i]
            n = np.linalg.norm(v)
            return v / n if n else v
    print("embedder: TF-IDF baseline (set a real embeddings endpoint for semantics)")
    return Tfidf

with get_session() as s:
    _all = s.query(Product).all()
products = [{"sku": p.sku, "name": p.name,
             "price_num": p.price_num or 0,
             "category": p.categories or "",
             "use_cases": p.use_cases or "",
             "text": (p.name or "") + " " + (p.categories or "")}
            for p in _all]
print("loaded products:", len(products))

embedder = make_embedder()
if isinstance(embedder, type):  # Tfidf class returned (not called)
    tfidf = embedder([p["text"] for p in products] + ["outdoor gate light warranty price"])
    def embed(texts):
        if isinstance(texts, str): texts = [texts]
        return np.stack([tfidf.vec(t) for t in texts])
else:
    embed = embedder


loaded products: 125
embedder: Azure OpenAI (per-string)


In [3]:
# 2. Helpers: cosine, local recall, history grounding
from sqlalchemy import or_

def cosine(a, b):
    n = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / n) if n else 0.0

def local_recall(category=None, max_price=None, k=10):
    """Stand-in for the MCP recall stage (no taxonomy needed)."""
    with get_session() as s:
        q = s.query(Product)
        if category:
            q = q.filter(or_(Product.categories.ilike(f"%{category}%"),
                             Product.name.ilike(f"%{category}%"),
                             Product.description.ilike(f"%{category}%")))
        if max_price:
            q = q.filter(Product.price_num <= max_price)
        rows = q.limit(k).all()
    return [{"sku": p.sku, "name": p.name, "price_num": p.price_num or 0,
             "category": p.categories or "",
             "use_cases": p.use_cases or "",
             "text": (p.name or "") + " " + (p.categories or "")}
            for p in rows]

def fetch_faq_chunks(max_chunks=30):
    """Fetch FAQ / policy chunks from Neo4j (:Chunk nodes)."""
    import os
    from neo4j import GraphDatabase
    uri = os.getenv("NEO4J_URI", "")
    user = os.getenv("NEO4J_USER", "")
    pw = os.getenv("NEO4J_PASSWORD", "")
    db = os.getenv("NEO4J_DATABASE", "")
    if not uri or not user or not pw:
        return None
    if uri.startswith("neo4j+s"):
        uri = "neo4j+ssc" + uri[7:]
    try:
        driver = GraphDatabase.driver(uri, auth=(user, pw))
        with driver.session(database=db) as session:
            rows = session.run(
                "MATCH (c:Chunk) RETURN c.title AS title, c.text AS text LIMIT $max",
                max=max_chunks
            )
            texts = [f"{r.get('title', 'FAQ')}: {r['text'][:500]}" for r in rows if r.get('text')]
        driver.close()
        return texts if texts else None
    except Exception as e:
        print(f"  [Neo4j FAQ fetch failed: {e}]")
        return None

def ground(history, utterance):
    """Concatenate recent turns -> a standalone, context-rich query."""
    return " ".join(history + [utterance])

print("helpers ready")


helpers ready


## Approach B — History-grounded dense product ANN (per-tenant embeddings)

Embeds the **grounded** query (history + utterance) and runs ANN over product
embeddings. A bare follow-up ("the cheaper one") has no signal alone, but grounded
with history it lands on the focused product. This is Inventaa's cosine retriever
with conversation context.


In [6]:
# 3. Dense product ANN + history grounding
P = products
P_emb = embed([p["text"] for p in P])

def ann_search(grounded_query, scope_skus=None, k=5):
    qe = embed(grounded_query)[0]
    cand = [p for p in P if (scope_skus is None or p["sku"] in scope_skus)]
    scored = [(cosine(P_emb[P.index(p)], qe), p) for p in cand]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in scored[:k]]

# Scenario: T1 show gate lights, T2 follow-up "the cheaper one"
T1_query = "outdoor gate lights under 1500"
history = [T1_query]
recall = local_recall(category="gate", max_price=1500, k=10)
scope_skus = [p["sku"] for p in recall[:3]]
print("T1 recall (top3 scope):", [(p["sku"], p["price_num"]) for p in recall[:3]])

bare = ann_search("the cheaper one", scope_skus=scope_skus)
print("Bare 'the cheaper one' top:", [p["name"][:25] for p in bare])

grounded = ground(history, "the cheaper one")
grounded_res = ann_search(grounded, scope_skus=scope_skus)
print("Grounded top:", [p["name"][:30] for p in grounded_res])
assert len(grounded_res) > 0, "grounded ANN returned nothing"
print("[OK] B. history grounding recovers the focused product for a bare follow-up")


T1 recall (top3 scope): [('BLK10C', 263), ('EXB12C', 391), ('HQ289', 426)]
Bare 'the cheaper one' top: ['Ora Bulkhead LED Light fo', 'Exla Bulkhead LED Light f', 'Chiti Gate Light Design L']
Grounded top: ['Chiti Gate Light Design LED Pi', 'Ora Bulkhead LED Light for Gar', 'Exla Bulkhead LED Light for Ga']
[OK] B. history grounding recovers the focused product for a bare follow-up


## Approach A — Cross-encoder reranker over hybrid recall + `scope_skus`

Reranks the recall set using a multi-signal score (semantic similarity + structured
price signal), constrained to `scope_skus` (the products already shown). Here the
"cross-encoder" is proxied by bi-encoder cosine; swap in `ms-marco-MiniLM` for
production. The point tested: comparative follow-ups reorder within the shown set.


In [7]:
# 4. Reranker + scope_skus
def rerank(candidates, grounded_emb, max_price_seen, alpha=0.6):
    scored = []
    for p in candidates:
        sim = cosine(P_emb[P.index(p)], grounded_emb)
        price_norm = 1 - (p["price_num"] / max_price_seen) if max_price_seen else 0.0
        score = alpha * sim + (1 - alpha) * price_norm   # similarity + cheaper => higher
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in scored]

ge = embed(ground(history, "the cheaper one"))[0]
max_price_seen = max((p["price_num"] for p in recall), default=1) or 1
reranked = rerank(recall, ge, max_price_seen)
print("Reranked (scope=shown set), cheapest-first signal:")
for p in reranked[:5]:
    print(f"   Rs.{p['price_num']:>6}  {p['name'][:35]}")

cheapest = min(recall, key=lambda p: p["price_num"])
print("\nCheapest in shown set:", cheapest["name"], "Rs.", cheapest["price_num"])
assert cheapest["sku"] in [p["sku"] for p in reranked[:3]], "cheapest not surfaced by reranker"
print("[OK] A. reranker surfaces the cheapest within scope_skus for a comparative follow-up")

narrow = rerank([p for p in recall if p["sku"] in scope_skus[:2]], ge, max_price_seen)
print("\nscope_skus=2 -> reranked count:", len(narrow), "(constrained to shown subset)")


Reranked (scope=shown set), cheapest-first signal:
   Rs.   263  Ora Bulkhead LED Light for Gardens
   Rs.   391  Exla Bulkhead LED Light for Gardens
   Rs.   426  Chiti Gate Light Design LED Pillar 
   Rs.   441  Emiko Main Gate Light LED Gate Ligh
   Rs.   566  Zenia ECO Main Gate Light LED Gate 

Cheapest in shown set: Ora Bulkhead LED Light for Gardens Rs. 263
[OK] A. reranker surfaces the cheapest within scope_skus for a comparative follow-up

scope_skus=2 -> reranked count: 2 (constrained to shown subset)


## Approach C — Structured NLU slot-filling (no taxonomy call)

Retrieval driven by extracted slots (category / price / filters), not a taxonomy
keyword match on raw text. Taxonomy becomes a validator only.


In [8]:
# 5. Slot-filling local retrieval (mirrors MCP intent_data path)
def slot_retrieve(intent_data):
    cat = intent_data.get("filters", {}).get("category")
    mp = intent_data.get("preferences", {}).get("max_price")
    return local_recall(category=cat, max_price=mp, k=10)

# Query with NO taxonomy words; slots carry the meaning.
slots = {"intent": "find_product",
         "filters": {"category": "gate"},
         "preferences": {"max_price": 1500}}
res = slot_retrieve(slots)
print("slot-driven products:", len(res))
assert len(res) > 0, "slot-filling returned 0"
assert all(p["price_num"] <= 1500 for p in res), "price slot not applied"
print("[OK] C. slots (category + max_price) retrieve correctly without a taxonomy call")


slot-driven products: 10
[OK] C. slots (category + max_price) retrieve correctly without a taxonomy call


## Approach D — Pick-feedback loop (commerce signal)

When the user picks a product, log it; future recall boosts the picked item (and
its siblings). Highest-leverage commerce signal; absent from the GraphRAG survey.


In [9]:
# 6. Pick-feedback loop (in-memory store)
feedback = {}  # session -> {sku: count}

def recall_with_feedback(query_emb, session, k=5, boost=0.25):
    scored = []
    for p in P:
        sim = cosine(P_emb[P.index(p)], query_emb)
        if session in feedback and p["sku"] in feedback[session]:
            sim += boost
        scored.append((sim, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in scored[:k]]

q = embed("gate light for my porch")[0]
before = recall_with_feedback(q, "sess1")
picked = before[0]["sku"]
feedback.setdefault("sess1", {})[picked] = 1

after = recall_with_feedback(q, "sess1")
print("before pick top:", before[0]["name"][:30])
print("after pick top:", after[0]["name"][:30], "| picked:", picked)
assert after[0]["sku"] == picked, "pick not boosted to top"
print("[OK] D. picked product is boosted in subsequent recall")


before pick top: Tacita Modern  12W Outdoor Gat
after pick top: Tacita Modern  12W Outdoor Gat | picked: 12M-2026B
[OK] D. picked product is boosted in subsequent recall


## Approach E — RAPTOR-style hierarchical retrieval for FAQ / policy

Build a 2-level index over FAQ chunks fetched from Neo4j `:Chunk` nodes:
leaf chunks + summary nodes (cluster centroids). Falls back to policy texts only
if Neo4j is unreachable. This mirrors the production hierarchical FAQ index.


In [10]:
# 7. RAPTOR-style 2-level retrieval over a doc corpus
policy_texts = [
    "All Inventaa products carry a 1-Year replacement warranty against manufacturing defects.",
    "Free delivery is available on orders above Rs. 999 within India.",
    "Returns are accepted within 7 days of delivery in original condition.",
]
faq_chunks = fetch_faq_chunks()
if faq_chunks and len(faq_chunks) >= 5:
    corpus = faq_chunks + policy_texts
    print(f"RAPTOR corpus: {len(corpus)} docs ({len(faq_chunks)} FAQ chunks + {len(policy_texts)} policies)")
else:
    corpus = policy_texts
    print(f"RAPTOR corpus: {len(corpus)} policy texts only (no FAQ chunks available)")

C_emb = embed(corpus)

k = min(5, len(corpus))
if k < 2:
    print("  corpus too small for RAPTOR clustering; using flat cosine search instead")
    def raptor_retrieve(query, top_clusters=2, top_leaves=3):
        qe = embed(query)[0]
        scored = sorted(range(len(corpus)), key=lambda i: cosine(qe, C_emb[i]), reverse=True)
        return [corpus[i] for i in scored[:top_leaves]]
else:
    rng = np.random.default_rng(0)
    centroids = C_emb[rng.choice(len(C_emb), k, replace=False)]
    for _ in range(5):  # a few Lloyd iterations
        assign = [int(np.argmin([1 - cosine(c, ctr) for ctr in centroids])) for c in C_emb]
        new = []
        for j in range(k):
            members = [C_emb[i] for i in range(len(C_emb)) if assign[i] == j]
            if members:
                new.append(np.mean(members, axis=0))
        if len(new) == k:
            centroids = np.stack(new)

    def raptor_retrieve(query, top_clusters=2, top_leaves=3):
        qe = embed(query)[0]
        cl = sorted(range(k), key=lambda j: cosine(qe, centroids[j]), reverse=True)[:top_clusters]
        assign = [int(np.argmin([1 - cosine(c, ctr) for ctr in centroids])) for c in C_emb]
        leaves = [i for i in range(len(corpus)) if assign[i] in cl]
        leaves.sort(key=lambda i: cosine(qe, C_emb[i]), reverse=True)
        return [corpus[i] for i in leaves[:top_leaves]]

print("RAPTOR retrieve 'warranty policy':")
for d in raptor_retrieve("what is the warranty policy"):
    print("  -", d[:80])
assert any("warranty" in d.lower() for d in raptor_retrieve("what is the warranty policy"))
print("[OK] E. RAPTOR hierarchical retrieval surfaces the relevant policy leaf")


RAPTOR corpus: 3 policy texts only (no FAQ chunks available)
RAPTOR retrieve 'warranty policy':
  - All Inventaa products carry a 1-Year replacement warranty against manufacturing 
  - Returns are accepted within 7 days of delivery in original condition.
[OK] E. RAPTOR hierarchical retrieval surfaces the relevant policy leaf


## Summary

| Approach | Implemented here | Production home |
|---|---|---|
| B. Dense product ANN + history | local ANN over SQLite products | per-tenant product embedding index |
| A. Cross-encoder rerank + scope_skus | local rerank + scope constraint | `search_catalog` rerank step + `scope_skus` param |
| C. Slot-filling | local SQLite slot query | agent passes `intent_data` (already done) |
| D. Pick-feedback | in-memory feedback boost | per-tenant feedback store |
| E. RAPTOR FAQ | 2-level cluster over Neo4j FAQ chunks | hierarchical index over `:Chunk`/`:FAQ` |

All five run locally. To ship, move the reranker / ANN / feedback / RAPTOR index
into the MCP server and wire `scope_skus`.
